In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Qiskit Chemistry
#A set of tools, algorithms and software for use with quantum computers
#to carry out research and investigate
#how to take advantage of the quantum computational power to solve chemistry problems.

#Approach
#1. Decompose the computation of the ground state energy of a molecule in 4 steps:
#Step 1 : Define a molecule and get integrals from a computational chemistry driver (PySCF in this case)
#Step 2 : Construct a Fermionic Hamiltonian and map it onto a qubit Hamiltonian
#Step 3 : Instantiate and initialize the algorithmic components,
#such as the quantum algorithm VQE, the optimizer and variational form it will use,
#and the initial_state to initialize the variational form
#Step 4 : Run the algorithm on a quantum backend and retrieve the results

%matplotlib inline
# Importing standard Qiskit libraries and configuring account
import numpy as np
from qiskit import QuantumCircuit
from qiskit_backend_config import execute, Aer, IBMQ
from qiskit.visualization import *
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

# Qiskit Aqua was removed: VQE and the exact eigensolver now come from qiskit-algorithms
from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA

# Loading your IBM Q account(s)
provider = IBMQ.load_account()


## Note: Steps 1 and 2 (LiH) need software that is not available here

`qiskit.chemistry` was removed (its successor is the separate **qiskit-nature** package), and the
PySCF driver used to compute the molecular integrals **does not support Windows**. The original LiH
steps are kept below as text for reference.

So that the rest of the workflow (Steps 3 and 4) still runs, the cell after them loads the
well-known 2-qubit Hamiltonian of **H2** (STO-3G basis, bond length 0.735 Angstrom, parity mapping
with two-qubit reduction) that these steps would produce for hydrogen.


```python
#Step 1: Define a molecule
#Example : we use LiH in the sto3g basis with the PySCF driver to get fermionic Hamiltonian
#The molecule object records the information from the PySCF driver.
driver = PySCFDriver(atom='Li .0 .0 .0; H .0 .0 1.6', unit=UnitsType.ANGSTROM,
                     charge=0, spin=0, basis='sto3g')
molecule = driver.run()
```


```python
#Step 2: Prepare qubit Hamiltonian
#Setup the to-be-frozen and to-be-removed orbitals to reduce the problem size 
#when we map to the qubit Hamiltonian. 
#Furthermore, we define the mapping type for the qubit Hamiltonian. 
#For the particular parity mapping, we can further reduce the problem size.

# please be aware that the idx here with respect to original idx
freeze_list = [0]
remove_list = [-3, -2] # negative number denotes the reverse order
map_type = 'parity'

h1 = molecule.one_body_integrals
h2 = molecule.two_body_integrals
nuclear_repulsion_energy = molecule.nuclear_repulsion_energy

num_particles = molecule.num_alpha + molecule.num_beta
num_spin_orbitals = molecule.num_orbitals * 2
print("HF energy: {}".format(molecule.hf_energy - molecule.nuclear_repulsion_energy))
print("# of electrons: {}".format(num_particles))
print("# of spin orbitals: {}".format(num_spin_orbitals))
```


```python
# prepare full idx of freeze_list and remove_list
# convert all negative idx to positive
remove_list = [x % molecule.num_orbitals for x in remove_list]
freeze_list = [x % molecule.num_orbitals for x in freeze_list]
# update the idx in remove_list of the idx after frozen, since the idx of orbitals are changed after freezing
remove_list = [x - len(freeze_list) for x in remove_list]
remove_list += [x + molecule.num_orbitals - len(freeze_list)  for x in remove_list]
freeze_list += [x + molecule.num_orbitals for x in freeze_list]

# prepare fermionic hamiltonian with orbital freezing and eliminating, and then map to qubit hamiltonian
# and if PARITY mapping is selected, reduction qubits
energy_shift = 0.0
qubit_reduction = True if map_type == 'parity' else False

ferOp = FermionicOperator(h1=h1, h2=h2)
if len(freeze_list) > 0:
    ferOp, energy_shift = ferOp.fermion_mode_freezing(freeze_list)
    num_spin_orbitals -= len(freeze_list)
    num_particles -= len(freeze_list)
if len(remove_list) > 0:
    ferOp = ferOp.fermion_mode_elimination(remove_list)
    num_spin_orbitals -= len(remove_list)

qubitOp = ferOp.mapping(map_type=map_type, threshold=0.00000001)
qubitOp = Z2Symmetries.two_qubit_reduction(qubitOp, num_particles) if qubit_reduction else qubitOp
qubitOp.chop(10**-10)

print(qubitOp.print_details())
print(qubitOp)
```


In [ ]:
#Steps 1+2 for H2: the qubit Hamiltonian (Hartree), as produced by the PySCF driver + parity mapping
qubitOp = SparsePauliOp.from_list([
    ("II", -1.052373245772859),
    ("IZ", 0.39793742484318045),
    ("ZI", -0.39793742484318045),
    ("ZZ", -0.01128010425623538),
    ("XX", 0.18093119978423156),
])
nuclear_repulsion_energy = 0.7199689944489797
energy_shift = 0.0          # no frozen orbitals for H2
num_particles = 2
num_spin_orbitals = 4
print(qubitOp)


In [ ]:
#We use the classical eigen decomposition to get the smallest eigenvalue as a reference.
# Using exact eigensolver to get the smallest eigenvalue
exact_eigensolver = NumPyMinimumEigensolver()
ret = exact_eigensolver.compute_minimum_eigenvalue(qubitOp)
print('The computed energy is: {:.12f}'.format(ret.eigenvalue.real))
print('The total ground state energy is: {:.12f}'.format(ret.eigenvalue.real + energy_shift + nuclear_repulsion_energy))


In [ ]:
#Step 3: Initiate and configure the instances
#To run VQE we require
#VQE algorithm
#Classical Optimizer
#A variational form (ansatz)

# the Estimator primitive replaces the statevector backend / QuantumInstance
estimator = StatevectorEstimator()

# setup COBYLA classical optimizer
max_eval = 200
cobyla = COBYLA(maxiter=max_eval)

# setup the HartreeFock state: in the parity mapping with two-qubit reduction H2's HF state is |01>
HF_state = QuantumCircuit(qubitOp.num_qubits)
HF_state.x(0)

# setup the variational form (qiskit.chemistry's UCCSD moved to qiskit-nature; a hardware-efficient
# RY/CX ansatz starting from the HF state reaches the exact ground state of this 2-qubit problem)
var_form = TwoLocal(qubitOp.num_qubits, 'ry', 'cx', reps=1, initial_state=HF_state)

# setup VQE
vqe = VQE(estimator, var_form, cobyla,
          initial_point=np.random.default_rng(7).uniform(-0.1, 0.1, var_form.num_parameters))


In [ ]:
#Step 4: Run algorithm and retrieve the results
results = vqe.compute_minimum_eigenvalue(qubitOp)
print('The computed ground state energy is: {:.12f}'.format(results.eigenvalue.real))
print('The total ground state energy is: {:.12f}'.format(results.eigenvalue.real + energy_shift + nuclear_repulsion_energy))
print("Parameters: {}".format(results.optimal_point))
